# Running simulations through the cloud 

This notebook is a tutorial of the API used for submitting simulations to our servers.


In [1]:
import tidy3d as td
import tidy3d.web as web

# set logging level to ERROR because we'll only running simulations to demonstrate the web API, we dont care about the results.
td.config.logging_level = "ERROR"


## Setup

Let's set up a simple simulation to get started.

In [2]:
# whether to print output in web functions, note: if False (default) they will all run silently
verbose = True

# set up parameters of simulation
dl = 0.05
pml = td.PML()
sim_size = [4, 4, 4]
freq0 = 3e14
fwidth = 1e13
run_time = 1 / fwidth

# create structure
dielectric = td.Medium.from_nk(n=2, k=0, freq=freq0)
square = td.Structure(
    geometry=td.Box(center=[0, 0, 0], size=[1.5, 1.5, 1.5]), medium=dielectric
)

# create source
source = td.UniformCurrentSource(
    center=(-1.5, 0, 0),
    size=(0, 0.4, 0.4),
    source_time=td.GaussianPulse(freq0=freq0, fwidth=fwidth),
    polarization="Ex",
)

# create monitor
monitor = td.FieldMonitor(
    fields=["Ex", "Ey", "Ez"],
    center=(0, 0, 0),
    size=(td.inf, td.inf, 0),
    freqs=[freq0],
    name="field",
)

# Initialize simulation
sim = td.Simulation(
    size=sim_size,
    grid_spec=td.GridSpec.uniform(dl),
    structures=[square],
    sources=[source],
    monitors=[monitor],
    run_time=run_time,
    boundary_spec=td.BoundarySpec.all_sides(boundary=pml),
)


## Running simulation manually

For the most control, you can run the simulation through the Tidy3D web API.
Each simulation running on the server is identified by a `task_id`, which must be specified in the web API.
Let's walk through submitting a simulation this way.

In [3]:
# upload the simulation to our servers
task_id = web.upload(sim, task_name="webAPI", verbose=verbose)

# start the simulation running
web.start(task_id)

# monitor the simulation, dont move on to next line until completed.
web.monitor(task_id, verbose=verbose)

# download the results and load into a simulation data object for plotting, post processing etc.
sim_data = web.load(task_id, path="data/sim.hdf5", verbose=verbose)


10:12:58 PDT Created task 'webAPI' with task_id                                 
             'fdve-1f8560a9-2d10-4400-9b39-d4b39c152077' and task_type 'FDTD'.

             View task using web UI at                                          
             ]8;id=22020;https://tidy3d.simulation.cloud/workbench?taskId=fdve-1f8560a9-2d10-4400-9b39-d4b39c152077\'https://tidy3d.simulation.cloud/workbench?]8;;\]8;id=387947;https://tidy3d.simulation.cloud/workbench?taskId=fdve-1f8560a9-2d10-4400-9b39-d4b39c152077\taskId]8;;\]8;id=22020;https://tidy3d.simulation.cloud/workbench?taskId=fdve-1f8560a9-2d10-4400-9b39-d4b39c152077\=]8;;\]8;id=149321;https://tidy3d.simulation.cloud/workbench?taskId=fdve-1f8560a9-2d10-4400-9b39-d4b39c152077\fdve]8;;\]8;id=22020;https://tidy3d.simulation.cloud/workbench?taskId=fdve-1f8560a9-2d10-4400-9b39-d4b39c152077\-1f8560a9-2d1]8;;\
             ]8;id=22020;https://tidy3d.simulation.cloud/workbench?taskId=fdve-1f8560a9-2d10-4400-9b39-d4b39c152077\0-4400-9b39-d4b39c152077']8;;\.

Output()

10:13:00 PDT status = queued

             To cancel the simulation, use 'web.abort(task_id)' or              
             'web.delete(task_id)' or abort/delete the task in the web UI.      
             Terminating the Python script will not stop the job running on the 
             cloud.

...

Output()

10:13:10 PDT status = success

10:13:11 PDT View simulation result at                                          
             ]8;id=97589;https://tidy3d.simulation.cloud/workbench?taskId=fdve-1f8560a9-2d10-4400-9b39-d4b39c152077\'https://tidy3d.simulation.cloud/workbench?]8;;\]8;id=685966;https://tidy3d.simulation.cloud/workbench?taskId=fdve-1f8560a9-2d10-4400-9b39-d4b39c152077\taskId]8;;\]8;id=97589;https://tidy3d.simulation.cloud/workbench?taskId=fdve-1f8560a9-2d10-4400-9b39-d4b39c152077\=]8;;\]8;id=814645;https://tidy3d.simulation.cloud/workbench?taskId=fdve-1f8560a9-2d10-4400-9b39-d4b39c152077\fdve]8;;\]8;id=97589;https://tidy3d.simulation.cloud/workbench?taskId=fdve-1f8560a9-2d10-4400-9b39-d4b39c152077\-1f8560a9-2d1]8;;\
             ]8;id=97589;https://tidy3d.simulation.cloud/workbench?taskId=fdve-1f8560a9-2d10-4400-9b39-d4b39c152077\0-4400-9b39-d4b39c152077']8;;\.

Output()

             loading simulation from data/sim.hdf5

While we broke down each of the individual steps above, one can also perform the entire process in one line by calling the [web.run()](https://docs.flexcompute.com/projects/tidy3d/en/latest/api/_autosummary/tidy3d.web.api.webapi.run.html) function as follows.

```python
sim_data = web.run(sim, task_name='webAPI', path='data/sim.hdf5')
```

(We won't run it again in this notebook to save time).

Sometimes this is more convenient, but other times it can be helpful to have the steps broken down one by one, for example if the simulation is long, you may want to [web.start](https://docs.flexcompute.com/projects/tidy3d/en/latest/api/_autosummary/tidy3d.web.api.webapi.start.html) and then exit the session and load the results at a later time using [web.load](https://docs.flexcompute.com/projects/tidy3d/en/latest/api/_autosummary/tidy3d.web.api.webapi.load.html).


## Job Container

The [web.Job](https://docs.flexcompute.com/projects/tidy3d/en/latest/api/_autosummary/tidy3d.web.api.container.Job.html) interface provides a more convenient way to manage single simuations, mainly because it eliminates the need for keeping track of the `task_id` and original [Simulation](https://docs.flexcompute.com/projects/tidy3d/en/latest/api/_autosummary/tidy3d.Simulation.html).

We can get the cost estimate of running the task before actually running it. This prevents us from accidentally running large jobs that we set up by mistake. The estimated cost is the maximum cost corresponding to running all the time steps.

In [4]:
# initializes job, puts task on server (but doesnt run it)
job = web.Job(simulation=sim, task_name="job", verbose=verbose)

# estimate the maximum cost
estimated_cost = web.estimate_cost(job.task_id)

10:13:12 PDT Created task 'job' with task_id                                    
             'fdve-acfa6b65-6852-42fc-a2ef-8c887425aedb' and task_type 'FDTD'.

             View task using web UI at                                          
             ]8;id=840638;https://tidy3d.simulation.cloud/workbench?taskId=fdve-acfa6b65-6852-42fc-a2ef-8c887425aedb\'https://tidy3d.simulation.cloud/workbench?]8;;\]8;id=111355;https://tidy3d.simulation.cloud/workbench?taskId=fdve-acfa6b65-6852-42fc-a2ef-8c887425aedb\taskId]8;;\]8;id=840638;https://tidy3d.simulation.cloud/workbench?taskId=fdve-acfa6b65-6852-42fc-a2ef-8c887425aedb\=]8;;\]8;id=856456;https://tidy3d.simulation.cloud/workbench?taskId=fdve-acfa6b65-6852-42fc-a2ef-8c887425aedb\fdve]8;;\]8;id=840638;https://tidy3d.simulation.cloud/workbench?taskId=fdve-acfa6b65-6852-42fc-a2ef-8c887425aedb\-acfa6b65-685]8;;\
             ]8;id=840638;https://tidy3d.simulation.cloud/workbench?taskId=fdve-acfa6b65-6852-42fc-a2ef-8c887425aedb\2-42fc-a2ef-8c887425aedb']8;;\.

Output()

             Maximum FlexCredit cost: 0.025. Minimum cost depends on task       
             execution details. Use 'web.real_cost(task_id)' to get the billed  
             FlexCredit cost after a simulation run.

While [Job](https://docs.flexcompute.com/projects/tidy3d/en/latest/api/_autosummary/tidy3d.web.api.container.Job.html) has methods with names identical to the web API functions above, which give some more granular control, it is often most convenient to call `.run()` so we will do that now.

In [5]:
# start job, monitor, and load results
sim_data = job.run(path="data/sim.hdf5")

10:13:13 PDT status = queued

             To cancel the simulation, use 'web.abort(task_id)' or              
             'web.delete(task_id)' or abort/delete the task in the web UI.      
             Terminating the Python script will not stop the job running on the 
             cloud.

Output()

10:13:18 PDT status = preprocess

10:13:19 PDT Maximum FlexCredit cost: 0.025. Use 'web.real_cost(task_id)' to get
             the billed FlexCredit cost after a simulation run.

...

Output()

10:13:24 PDT status = success

             View simulation result at                                          
             ]8;id=507328;https://tidy3d.simulation.cloud/workbench?taskId=fdve-acfa6b65-6852-42fc-a2ef-8c887425aedb\'https://tidy3d.simulation.cloud/workbench?]8;;\]8;id=667766;https://tidy3d.simulation.cloud/workbench?taskId=fdve-acfa6b65-6852-42fc-a2ef-8c887425aedb\taskId]8;;\]8;id=507328;https://tidy3d.simulation.cloud/workbench?taskId=fdve-acfa6b65-6852-42fc-a2ef-8c887425aedb\=]8;;\]8;id=878539;https://tidy3d.simulation.cloud/workbench?taskId=fdve-acfa6b65-6852-42fc-a2ef-8c887425aedb\fdve]8;;\]8;id=507328;https://tidy3d.simulation.cloud/workbench?taskId=fdve-acfa6b65-6852-42fc-a2ef-8c887425aedb\-acfa6b65-685]8;;\
             ]8;id=507328;https://tidy3d.simulation.cloud/workbench?taskId=fdve-acfa6b65-6852-42fc-a2ef-8c887425aedb\2-42fc-a2ef-8c887425aedb']8;;\.

Output()

             loading simulation from data/sim.hdf5

Another convenient thing about [Job](https://docs.flexcompute.com/projects/tidy3d/en/latest/api/_autosummary/tidy3d.web.api.container.Job.html) objects is that they can be saved and loaded just like other Tidy3d components.

This is convenient if you want to save and load up the results of a job that has already finished, without needing to know the `task_id`.

In [6]:
# saves the job metadata to a single file
job.to_file("data/job.json")

# can exit session, break here, or continue in new session.

# load the job metadata from file
job_loaded = web.Job.from_file("data/job.json")

# download the data from the server and load it into a SimulationData object.
sim_data = job_loaded.load(path="data/sim.hdf5")


Output()

10:13:25 PDT loading simulation from data/sim.hdf5

## Batch Processing

Commonly one needs to submit a batch of Simulations.
One way to approach this using the web API is to upload, start, monitor, and load a series of tasks one by one, but this is clumsy and you can lose your data if the session gets interrupted.

A better way is to use the built-in [Batch](https://docs.flexcompute.com/projects/tidy3d/en/latest/api/_autosummary/tidy3d.web.api.container.Batch.html) object.
The batch object is like a [Job](https://docs.flexcompute.com/projects/tidy3d/en/latest/api/_autosummary/tidy3d.web.api.container.Job.html), but stores task metadata for a series of simulations.

In [7]:
# make a dictionary of  {task name : simulation} for demonstration
sims = {f"sim_{i}": sim for i in range(3)}

# initialize a batch and run them all
batch = web.Batch(simulations=sims, verbose=verbose)

# run the batch and store all of the data in the `data/` dir.
batch_results = batch.run(path_dir="data")


             Created task 'sim_0' with task_id                                  
             'fdve-0ac65882-4505-43dc-af7e-2161cf9f22c5' and task_type 'FDTD'.

             View task using web UI at                                          
             ]8;id=444597;https://tidy3d.simulation.cloud/workbench?taskId=fdve-0ac65882-4505-43dc-af7e-2161cf9f22c5\'https://tidy3d.simulation.cloud/workbench?]8;;\]8;id=349313;https://tidy3d.simulation.cloud/workbench?taskId=fdve-0ac65882-4505-43dc-af7e-2161cf9f22c5\taskId]8;;\]8;id=444597;https://tidy3d.simulation.cloud/workbench?taskId=fdve-0ac65882-4505-43dc-af7e-2161cf9f22c5\=]8;;\]8;id=100181;https://tidy3d.simulation.cloud/workbench?taskId=fdve-0ac65882-4505-43dc-af7e-2161cf9f22c5\fdve]8;;\]8;id=444597;https://tidy3d.simulation.cloud/workbench?taskId=fdve-0ac65882-4505-43dc-af7e-2161cf9f22c5\-0ac65882-450]8;;\
             ]8;id=444597;https://tidy3d.simulation.cloud/workbench?taskId=fdve-0ac65882-4505-43dc-af7e-2161cf9f22c5\5-43dc-af7e-2161cf9f22c5']8;;\.

Output()

10:13:26 PDT Created task 'sim_1' with task_id                                  
             'fdve-bc71441e-10eb-4229-8eb1-e6cb597d09a7' and task_type 'FDTD'.

             View task using web UI at                                          
             ]8;id=161021;https://tidy3d.simulation.cloud/workbench?taskId=fdve-bc71441e-10eb-4229-8eb1-e6cb597d09a7\'https://tidy3d.simulation.cloud/workbench?]8;;\]8;id=735581;https://tidy3d.simulation.cloud/workbench?taskId=fdve-bc71441e-10eb-4229-8eb1-e6cb597d09a7\taskId]8;;\]8;id=161021;https://tidy3d.simulation.cloud/workbench?taskId=fdve-bc71441e-10eb-4229-8eb1-e6cb597d09a7\=]8;;\]8;id=302127;https://tidy3d.simulation.cloud/workbench?taskId=fdve-bc71441e-10eb-4229-8eb1-e6cb597d09a7\fdve]8;;\]8;id=161021;https://tidy3d.simulation.cloud/workbench?taskId=fdve-bc71441e-10eb-4229-8eb1-e6cb597d09a7\-bc71441e-10e]8;;\
             ]8;id=161021;https://tidy3d.simulation.cloud/workbench?taskId=fdve-bc71441e-10eb-4229-8eb1-e6cb597d09a7\b-4229-8eb1-e6cb597d09a7']8;;\.

...

10:13:28 PDT Started working on Batch.

10:13:29 PDT Maximum FlexCredit cost: 0.075 for the whole batch.

             Use 'Batch.real_cost()' to get the billed FlexCredit cost after the
             Batch has completed.

Output()

10:13:40 PDT Batch complete.

When the batch is completed, the output is not a [SimulationData](https://docs.flexcompute.com/projects/tidy3d/en/latest/api/_autosummary/tidy3d.SimulationData.html) but rather a [BatchData](https://docs.flexcompute.com/projects/tidy3d/en/latest/api/_autosummary/tidy3d.web.api.container.BatchData.html).  The data within this [BatchData](https://docs.flexcompute.com/projects/tidy3d/en/latest/api/_autosummary/tidy3d.web.api.container.BatchData.html) object can either be indexed directly `batch_results[task_name]` or can be looped through `batch_results.items()` to get the [SimulationData](https://docs.flexcompute.com/projects/tidy3d/en/latest/api/_autosummary/tidy3d.SimulationData.html) for each task.

This was chosen to reduce the memory strain from loading all [SimulationData](https://docs.flexcompute.com/projects/tidy3d/en/latest/api/_autosummary/tidy3d.SimulationData.html) objects at once.

Alternatively, the batch can be looped through (several times) using the `.items()` method, similar to a dictionary.

In [8]:
# grab the sum of intensities in the simulation one by one (to save memory)
intensities = {}
for task_name, sim_data in batch_results.items():
    intensity = sim_data.get_intensity("field").sel(f=freq0)
    sum_intensity = float(intensity.sum(("x", "y")).values[0])
    intensities[task_name] = sum_intensity

print(intensities)


Output()

10:13:41 PDT loading simulation from                                            
             data/fdve-0ac65882-4505-43dc-af7e-2161cf9f22c5.hdf5

Output()

             loading simulation from                                            
             data/fdve-bc71441e-10eb-4229-8eb1-e6cb597d09a7.hdf5

Output()

10:13:42 PDT loading simulation from                                            
             data/fdve-895d04fb-188f-44df-8465-d4d364f4989a.hdf5

{'sim_0': 6377911.0, 'sim_1': 6377911.0, 'sim_2': 6377911.0}


## Simulation Batching

Finally, one perform batch processing of several simulations in a single function call.

For this purpose, a [web.run_async](https://docs.flexcompute.com/projects/tidy3d/en/latest/api/_autosummary/tidy3d.web.api.asynchronous.run_async.html) function is provided, which works like the regular `web.run` but accepts a dictionary of simulations. 

Here is the previous example repeated using this feature.

In [9]:
batch_results = web.run_async(simulations=sims, verbose=verbose)


             Created task 'sim_0' with task_id                                  
             'fdve-ef1ac2ea-906f-42c6-9d40-983ac18bc94e' and task_type 'FDTD'.

             View task using web UI at                                          
             ]8;id=370994;https://tidy3d.simulation.cloud/workbench?taskId=fdve-ef1ac2ea-906f-42c6-9d40-983ac18bc94e\'https://tidy3d.simulation.cloud/workbench?]8;;\]8;id=749045;https://tidy3d.simulation.cloud/workbench?taskId=fdve-ef1ac2ea-906f-42c6-9d40-983ac18bc94e\taskId]8;;\]8;id=370994;https://tidy3d.simulation.cloud/workbench?taskId=fdve-ef1ac2ea-906f-42c6-9d40-983ac18bc94e\=]8;;\]8;id=163935;https://tidy3d.simulation.cloud/workbench?taskId=fdve-ef1ac2ea-906f-42c6-9d40-983ac18bc94e\fdve]8;;\]8;id=370994;https://tidy3d.simulation.cloud/workbench?taskId=fdve-ef1ac2ea-906f-42c6-9d40-983ac18bc94e\-ef1ac2ea-906]8;;\
             ]8;id=370994;https://tidy3d.simulation.cloud/workbench?taskId=fdve-ef1ac2ea-906f-42c6-9d40-983ac18bc94e\f-42c6-9d40-983ac18bc94e']8;;\.

Output()

             Created task 'sim_1' with task_id                                  
             'fdve-d3293bbc-340c-44a3-845b-dd9b5d63ab19' and task_type 'FDTD'.

             View task using web UI at                                          
             ]8;id=68812;https://tidy3d.simulation.cloud/workbench?taskId=fdve-d3293bbc-340c-44a3-845b-dd9b5d63ab19\'https://tidy3d.simulation.cloud/workbench?]8;;\]8;id=319247;https://tidy3d.simulation.cloud/workbench?taskId=fdve-d3293bbc-340c-44a3-845b-dd9b5d63ab19\taskId]8;;\]8;id=68812;https://tidy3d.simulation.cloud/workbench?taskId=fdve-d3293bbc-340c-44a3-845b-dd9b5d63ab19\=]8;;\]8;id=863269;https://tidy3d.simulation.cloud/workbench?taskId=fdve-d3293bbc-340c-44a3-845b-dd9b5d63ab19\fdve]8;;\]8;id=68812;https://tidy3d.simulation.cloud/workbench?taskId=fdve-d3293bbc-340c-44a3-845b-dd9b5d63ab19\-d3293bbc-340]8;;\
             ]8;id=68812;https://tidy3d.simulation.cloud/workbench?taskId=fdve-d3293bbc-340c-44a3-845b-dd9b5d63ab19\c-44a3-845b-dd9b5d63ab19']8;;\.

...

10:13:45 PDT Started working on Batch.

10:13:46 PDT Maximum FlexCredit cost: 0.075 for the whole batch.

             Use 'Batch.real_cost()' to get the billed FlexCredit cost after the
             Batch has completed.

Output()

10:13:56 PDT Batch complete.

In [10]:
# grab the sum of intensities in the simulation one by one (to save memory)
intensities = {}
for task_name, sim_data in batch_results.items():
    intensity = sim_data.get_intensity("field").sel(f=freq0)
    sum_intensity = float(intensity.sum(("x", "y")).values[0])
    intensities[task_name] = sum_intensity

print(intensities)


Output()

10:13:57 PDT loading simulation from                                            
             ./fdve-ef1ac2ea-906f-42c6-9d40-983ac18bc94e.hdf5

Output()

10:13:58 PDT loading simulation from                                            
             ./fdve-d3293bbc-340c-44a3-845b-dd9b5d63ab19.hdf5

Output()

             loading simulation from                                            
             ./fdve-dc780457-f9f5-463e-8ed6-59ff2e0115de.hdf5

{'sim_0': 6377911.0, 'sim_1': 6377911.0, 'sim_2': 6377911.0}


After going through this tutorial, you have learned how to run simulations with Tidy3D via the web API. If you are new to Tidy3D or the finite-difference time-domain (FDTD) method, we highly recommend going through our [FDTD101](https://www.flexcompute.com/fdtd101/) tutorials and [example library](https://www.flexcompute.com/tidy3d/examples/) before starting your own simulation adventure. 